In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df_inventory = spark.read.parquet("abfss://retail_project7@onelake.dfs.fabric.microsoft.com/Retail_LH.Lakehouse/Files/bronze/inventory_data.parquet")

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 3, Finished, Available, Finished, False)

In [2]:
display(df_inventory)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 04ba1f0a-f6f4-4a3c-9d37-f7814f0225d0)

In [3]:
df_orders = spark.read.parquet("abfss://retail_project7@onelake.dfs.fabric.microsoft.com/Retail_LH.Lakehouse/Files/bronze/orders_data.parquet")

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 5, Finished, Available, Finished, False)

In [4]:
display(df_orders)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b85d22b1-1483-40e1-bbec-18738f40c6af)

In [5]:
df_returns = spark.read.parquet("abfss://retail_project7@onelake.dfs.fabric.microsoft.com/Retail_LH.Lakehouse/Files/bronze/returns_data.xlsx.parquet")

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 7, Finished, Available, Finished, False)

In [6]:
display(df_returns)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 94173760-bde1-417a-8569-8d7d0564b230)

In [7]:
from pyspark.sql.functions import *

df_inventory_silver = (
    df_inventory
    .withColumnRenamed("productName", "product_name")
    .withColumnRenamed("last_stocked", "last_stocked")
    .withColumnRenamed("product_id", "product_id")
)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 9, Finished, Available, Finished, False)

In [8]:
from pyspark.sql.functions import *

df_inventory_silver = (
    df_inventory_silver
    .withColumn(
        "stock",
        when(col("stock").rlike("^[0-9]+$"), col("stock").cast("int"))
        .when(col("stock").isNull() | (trim(col("stock")) == ""), lit(None))
        .when(lower(col("stock")).contains("twenty five"), lit(25))
        .when(lower(col("stock")).contains("twenty"), lit(20))
        .when(lower(col("stock")).contains("eighteen"), lit(18))
        .when(lower(col("stock")).contains("fifteen"), lit(15))
        .when(lower(col("stock")).contains("twelve"), lit(12))
        .otherwise(lit(None))
        .cast("int")
    )
)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 10, Finished, Available, Finished, False)

In [9]:
display(df_inventory_silver)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a9278d4d-c65e-4117-a8d8-31697832e07e)

In [10]:
df_inventory_silver.columns

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 12, Finished, Available, Finished, False)

['product_id',
 'product_name',
 'stock',
 'last_stocked',
 'warehouse',
 'cost_price',
 'available']

In [11]:
from pyspark.sql.types import IntegerType, DoubleType

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 13, Finished, Available, Finished, False)

In [12]:
df_inventory_silver = (
    df_inventory_silver
    .withColumn(
        "stock",
        when(col("stock").rlike("^[0-9]+$"), col("stock").cast(IntegerType()))
        .when(col("stock").isNull() | (col("stock") == ""), lit(None))
        .otherwise(
            when(col("stock").rlike(".*twenty five.*"), lit(25))
            .when(col("stock").rlike(".*twenty.*"), lit(20))
            .when(col("stock").rlike(".*eighteen.*"), lit(18))
            .when(col("stock").rlike(".*fifteen.*"), lit(15))
            .when(col("stock").rlike(".*twelve.*"), lit(12))
            .otherwise(lit(None))
        ).cast(IntegerType())
    )
)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 14, Finished, Available, Finished, False)

In [13]:
display(df_inventory_silver)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bf2154eb-862d-4f1e-a52f-68519d6de73c)

In [20]:
from pyspark.sql.functions import *

df_inventory_silver = (
    df_inventory_silver
    .drop("last_stocked")
    .join(
        df_inventory.select("product_id", "last_stocked"),
        on="product_id",
        how="left"
    )
    .withColumn(
        "last_stocked",
        coalesce(
            to_date(col("last_stocked"), "yyyy/MM/dd"),
            to_date(col("last_stocked"), "dd-MM-yyyy"),
            to_date(col("last_stocked"), "yyyy.MM.dd"),
            to_date(col("last_stocked"), "yyyy-MM-dd"),
            to_date(col("last_stocked"), "dd.MM.yyyy"),
            to_date(col("last_stocked"), "dd/MM/yyyy"),
            to_date(col("last_stocked"), "MM/dd/yyyy")
        )
    )
)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 22, Finished, Available, Finished, False)

In [21]:
display(df_inventory_silver)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bb44cd97-8e9a-49b8-907a-bd85a8d1fbcd)

In [22]:
df_inventory_silver = (
    df_inventory_silver
    .withColumn(
        "cost_price",
        regexp_extract(
            col("cost_price"),
            r"(\d+\.?\d*)",
            1
        ).cast(DoubleType())
    )
)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 24, Finished, Available, Finished, False)

In [23]:
df_inventory_silver = (
    df_inventory_silver
    .withColumn(
        "warehouse",
        initcap(
            trim(
                regexp_replace(
                    col("warehouse"),
                    r"[^a-zA-Z0-9\s]",
                    " "
                )
            )
        )
    )
)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 25, Finished, Available, Finished, False)

In [24]:
display(df_inventory_silver)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aef6fd4a-271e-4d74-be12-4a055247f57e)

In [25]:
df_inventory_silver = (
    df_inventory_silver
    .withColumn(
        "available",
        when(
            lower(col("available")).isin("yes", "y", "true"),
            lit(True)
        )
        .when(
            lower(col("available")).isin("no", "n", "false"),
            lit(False)
        )
        .otherwise(None)
    )
)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 27, Finished, Available, Finished, False)

In [26]:
display(df_inventory_silver)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c2658d4d-7e8e-4a8a-8d9b-d3b3fe500265)

In [27]:
df_inventory_silver.printSchema()

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 29, Finished, Available, Finished, False)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- stock: integer (nullable = true)
 |-- warehouse: string (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- available: boolean (nullable = true)
 |-- last_stocked: date (nullable = true)



In [28]:
df_inventory_silver.write.mode("overwrite").format("delta").saveAsTable("silver_inventory")

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 30, Finished, Available, Finished, False)

In [29]:
from pyspark.sql.types import IntegerType, DoubleType

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 31, Finished, Available, Finished, False)

In [30]:
df_returns = (
    df_return_raw

    # Standardize column names
    .withColumnRenamed("Return_ID", "ReturnID")
    .withColumnRenamed("Order_ID", "OrderID")
    .withColumnRenamed("Customer_ID", "CustomerID")
    .withColumnRenamed("Return_Reason", "ReturnReason")
    .withColumnRenamed("Return_Date", "ReturnDate")
    .withColumnRenamed("Refund_Status", "RefundStatus")
    .withColumnRenamed("Pickup_Address", "PickupAddress")
    .withColumnRenamed("Return_Amount", "ReturnAmount")

    # Clean ReturnDate
    .withColumn(
        "ReturnDate",
        to_date(
            regexp_replace("ReturnDate", r"[./]", "-"),
            "dd-MM-yyyy"
        )
    )

    # Clean RefundStatus
    .withColumn(
        "RefundStatus",
        lower(
            regexp_replace(
                col("RefundStatus"),
                r"[^a-zA-Z]",
                ""
            )
        )
    )

    # Clean ReturnAmount
    .withColumn(
        "ReturnAmount",
        regexp_extract(
            col("ReturnAmount"),
            r"(\d+\.?\d*)",
            1
        ).cast(DoubleType())
    )

    # Clean PickupAddress
    .withColumn(
        "PickupAddress",
        initcap(
            trim(
                regexp_replace(
                    col("PickupAddress"),
                    r"[^a-zA-Z0-9\s]",
                    " "
                )
            )
        )
    )

    # Clean Product
    .withColumn(
        "Product",
        initcap(
            trim(
                regexp_replace(
                    col("Product"),
                    r"[^a-zA-Z0-9\s]",
                    ""
                )
            )
        )
    )

    # Clean CustomerID
    .withColumn(
        "CustomerID",
        trim(upper(col("CustomerID")))
    )

    # Remove rows with missing ReturnID
    .filter(col("ReturnID").isNotNull())
)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 32, Finished, Available, Finished, False)

NameError: name 'df_return_raw' is not defined

In [31]:
[x for x in globals() if x.startswith("df")]

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 33, Finished, Available, Finished, False)

['df_inventory', 'df_orders', 'df_returns', 'df_inventory_silver']

In [32]:
df_return_raw = df_returns

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 34, Finished, Available, Finished, False)

In [33]:
df_return_raw.columns

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 35, Finished, Available, Finished, False)

['Prop_0',
 'Prop_1',
 'Prop_2',
 'Prop_3',
 'Prop_4',
 'Prop_5',
 'Prop_6',
 'Prop_7',
 'Prop_8']

In [34]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

df_return_cleaned = (
    df_returns

    # Standardize column names
    .withColumnRenamed("Return_ID", "ReturnID")
    .withColumnRenamed("Order_ID", "OrderID")
    .withColumnRenamed("Customer_ID", "CustomerID")
    .withColumnRenamed("Return_Reason", "ReturnReason")
    .withColumnRenamed("Return_Date", "ReturnDate")
    .withColumnRenamed("Refund_Status", "RefundStatus")
    .withColumnRenamed("Pickup_Address", "PickupAddress")
    .withColumnRenamed("Return_Amount", "ReturnAmount")

    # Clean ReturnDate
    .withColumn(
        "ReturnDate",
        to_date(
            regexp_replace("ReturnDate", r"[./]", "-"),
            "dd-MM-yyyy"
        )
    )

    # Clean RefundStatus
    .withColumn(
        "RefundStatus",
        lower(
            regexp_replace(
                col("RefundStatus"),
                r"[^a-zA-Z]",
                ""
            )
        )
    )

    # Clean ReturnAmount
    .withColumn(
        "ReturnAmount",
        regexp_extract(
            col("ReturnAmount"),
            r"(\d+\.?\d*)",
            1
        ).cast(DoubleType())
    )

    # Clean PickupAddress
    .withColumn(
        "PickupAddress",
        initcap(
            trim(
                regexp_replace(
                    col("PickupAddress"),
                    r"[^a-zA-Z0-9\s]",
                    " "
                )
            )
        )
    )

    # Clean Product
    .withColumn(
        "Product",
        initcap(
            trim(
                regexp_replace(
                    col("Product"),
                    r"[^a-zA-Z0-9\s]",
                    ""
                )
            )
        )
    )

    # Clean CustomerID
    .withColumn(
        "CustomerID",
        trim(upper(col("CustomerID")))
    )

    # Remove rows where ReturnID is null
    .filter(col("ReturnID").isNotNull())
)

# Display cleaned Returns data
display(df_return_cleaned)

# Save to Silver Delta table
df_return_cleaned.write.mode("overwrite").format("delta").saveAsTable("silver_returns")

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 36, Finished, Available, Finished, False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `ReturnDate` cannot be resolved. Did you mean one of the following? [`Prop_0`, `Prop_1`, `Prop_2`, `Prop_3`, `Prop_4`].;
'Project [Prop_0#863, Prop_1#864, Prop_2#865, Prop_3#866, Prop_4#867, Prop_5#868, Prop_6#869, Prop_7#870, Prop_8#871, to_date(regexp_replace('ReturnDate, [./], -, 1), Some(dd-MM-yyyy), Some(UTC), false) AS ReturnDate#2041]
+- Relation [Prop_0#863,Prop_1#864,Prop_2#865,Prop_3#866,Prop_4#867,Prop_5#868,Prop_6#869,Prop_7#870,Prop_8#871] parquet


In [35]:
df_returns.show(10, truncate=False)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 37, Finished, Available, Finished, False)

+---------+--------+-----------+------------------+----------------+------------+-------------+--------------------+-------------+
|Prop_0   |Prop_1  |Prop_2     |Prop_3            |Prop_4          |Prop_5      |Prop_6       |Prop_7              |Prop_8       |
+---------+--------+-----------+------------------+----------------+------------+-------------+--------------------+-------------+
|Return_ID|Order_ID|Customer_ID|Product           |Return_Reason   |Return_Date |Refund_Status|Pickup_Address      |Return_Amount|
|R001     |1001    |C001       |apple iphone13    |Defective       |15-07-2023  |Done         |221B Baker St London|$799         |
|R002     |1002    |C_002      |samsung GALAXY s21|CHANGED mind    |2023/07/20  |Processing   |Sector-21 Noida     |₹1200        |
|R003     |1004    |C004       |oneplus NORD      |Late Delivery   |2023.07.22  |in progress  |mumbai              |499          |
|R004     |1005    |NULL       |nokia 3310        |not as described|20.07.2023  |Co

In [36]:
df_returns.columns

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 38, Finished, Available, Finished, False)

['Prop_0',
 'Prop_1',
 'Prop_2',
 'Prop_3',
 'Prop_4',
 'Prop_5',
 'Prop_6',
 'Prop_7',
 'Prop_8']

In [37]:
display(df_returns)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 37f51934-4195-43f1-8e29-f6e6de746dd3)

In [38]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

df_return_cleaned = (
    df_returns

    # Rename Prop columns using the actual header row
    .withColumnRenamed("Prop_0", "ReturnID")
    .withColumnRenamed("Prop_1", "OrderID")
    .withColumnRenamed("Prop_2", "CustomerID")
    .withColumnRenamed("Prop_3", "Product")
    .withColumnRenamed("Prop_4", "ReturnReason")
    .withColumnRenamed("Prop_5", "ReturnDate")
    .withColumnRenamed("Prop_6", "RefundStatus")
    .withColumnRenamed("Prop_7", "PickupAddress")
    .withColumnRenamed("Prop_8", "ReturnAmount")

    # Remove the original header row
    .filter(col("ReturnID") != "Return_ID")

    # Clean Return Date - handle all formats in your data
    .withColumn(
        "ReturnDate",
        coalesce(
            to_date(col("ReturnDate"), "dd-MM-yyyy"),
            to_date(col("ReturnDate"), "yyyy/MM/dd"),
            to_date(col("ReturnDate"), "yyyy.MM.dd"),
            to_date(col("ReturnDate"), "yyyy-MM-dd"),
            to_date(col("ReturnDate"), "dd.MM.yyyy"),
            to_date(col("ReturnDate"), "MM/dd/yyyy"),
            to_date(col("ReturnDate"), "MMMM d yyyy"),
            to_date(col("ReturnDate"), "MMM d yyyy")
        )
    )

    # Clean Refund Status
    .withColumn(
        "RefundStatus",
        lower(
            regexp_replace(
                col("RefundStatus"),
                r"[^a-zA-Z]",
                ""
            )
        )
    )

    # Clean Return Amount
    .withColumn(
        "ReturnAmount",
        regexp_extract(
            col("ReturnAmount"),
            r"(\d+\.?\d*)",
            1
        ).cast(DoubleType())
    )

    # Clean Pickup Address
    .withColumn(
        "PickupAddress",
        initcap(
            trim(
                regexp_replace(
                    col("PickupAddress"),
                    r"[^a-zA-Z0-9\s]",
                    " "
                )
            )
        )
    )

    # Clean Product
    .withColumn(
        "Product",
        initcap(
            trim(
                regexp_replace(
                    col("Product"),
                    r"[^a-zA-Z0-9\s]",
                    ""
                )
            )
        )
    )

    # Clean Customer ID
    .withColumn(
        "CustomerID",
        trim(upper(col("CustomerID")))
    )

    # Remove rows with missing ReturnID
    .filter(col("ReturnID").isNotNull())
)

display(df_return_cleaned)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 91cca6b8-a5af-44d3-b10d-898dd384e521)

In [39]:
df_return_cleaned.write.mode("overwrite").format("delta").saveAsTable("silver_returns")

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 41, Finished, Available, Finished, False)

In [40]:
df_orders.columns

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 42, Finished, Available, Finished, False)

['Order_ID',
 'cust_id',
 'Product_Name',
 'Qty',
 'Order_Date',
 'Order_Amount$',
 'Delivery_Status',
 'Payment_Mode',
 'Ship_Address',
 'Email',
 'Promo_Code',
 'Feedback_Score']

In [41]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

df_orders_cleaned = (
    df_orders

    # Standardize column names
    .withColumnRenamed("cust_id", "CustomerID")
    .withColumnRenamed("Product_Name", "ProductName")
    .withColumnRenamed("Order_Amount$", "OrderAmount")

    # Clean Order Date
    .withColumn(
        "Order_Date",
        to_date(
            regexp_replace(
                col("Order_Date"),
                "/",
                "-"
            ),
            "dd-MM-yyyy"
        )
    )

    # Standardize Customer ID
    .withColumn(
        "CustomerID",
        trim(upper(col("CustomerID")))
    )

    # Clean Email
    .withColumn(
        "Email",
        lower(trim(col("Email")))
    )

    # Handle empty Payment Mode
    .withColumn(
        "Payment_Mode",
        when(
            length(trim(col("Payment_Mode"))) == 0,
            "unknown"
        ).otherwise(col("Payment_Mode"))
    )

    # Replace null Promo Codes
    .withColumn(
        "Promo_Code",
        coalesce(
            col("Promo_Code"),
            lit("NO_PROMO")
        )
    )

    # Clean Order Amount
    .withColumn(
        "OrderAmount",
        abs(
            regexp_extract(
                col("OrderAmount"),
                r"(\d+\.?\d*)",
                1
            ).cast(DoubleType())
        )
    )

    # Add Year and Month
    .withColumn(
        "Order_Year",
        year("Order_Date")
    )
    .withColumn(
        "Order_Month",
        month("Order_Date")
    )

    # Create hash key
    .withColumn(
        "Order_Hash",
        sha2(
            concat_ws("|", *df_orders.columns),
            256
        )
    )

    # Remove rows with critical missing values
    .dropna(
        subset=[
            "Order_ID",
            "OrderAmount",
            "CustomerID"
        ]
    )

    # Remove duplicate Order IDs
    .dropDuplicates(["Order_ID"])
)

display(df_orders_cleaned)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 43, Finished, Available, Finished, False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `cust_id` cannot be resolved. Did you mean one of the following? [`Qty`, `Email`, `CustomerID`, `Order_ID`, `Promo_Code`].;
'Project [Order_ID#777, CustomerID#3165, ProductName#3126, Qty#780, Order_Date#3152, OrderAmount#3217, Delivery_Status#783, Payment_Mode#3191, Ship_Address#785, Email#3178, Promo_Code#3204, Feedback_Score#788, Order_Year#3230, Order_Month#3244, sha2(concat_ws(|, Order_ID#777, 'cust_id, 'Product_Name, Qty#780, Order_Date#3152, 'Order_Amount$, Delivery_Status#783, Payment_Mode#3191, Ship_Address#785, Email#3178, Promo_Code#3204, Feedback_Score#788), 256) AS Order_Hash#3259]
+- Project [Order_ID#777, CustomerID#3165, ProductName#3126, Qty#780, Order_Date#3152, OrderAmount#3217, Delivery_Status#783, Payment_Mode#3191, Ship_Address#785, Email#3178, Promo_Code#3204, Feedback_Score#788, Order_Year#3230, month(Order_Date#3152) AS Order_Month#3244]
   +- Project [Order_ID#777, C

In [42]:
from pyspark.sql.functions import *

df_orders_cleaned = (
    df_orders

    # Clean Customer ID
    .withColumn(
        "CustomerID",
        trim(upper(col("CustomerID")))
    )

    # Clean Email
    .withColumn(
        "Email",
        lower(trim(col("Email")))
    )

    # Handle empty Payment Mode
    .withColumn(
        "Payment_Mode",
        when(
            length(trim(col("Payment_Mode"))) == 0,
            "unknown"
        ).otherwise(col("Payment_Mode"))
    )

    # Replace null Promo Codes
    .withColumn(
        "Promo_Code",
        coalesce(
            col("Promo_Code"),
            lit("NO_PROMO")
        )
    )

    # Make Order Amount positive
    .withColumn(
        "OrderAmount",
        abs(col("OrderAmount").cast("double"))
    )

    # Add Year and Month
    .withColumn(
        "Order_Year",
        year("Order_Date")
    )
    .withColumn(
        "Order_Month",
        month("Order_Date")
    )

    # Remove rows with critical missing values
    .dropna(
        subset=[
            "Order_ID",
            "OrderAmount",
            "CustomerID"
        ]
    )

    # Remove duplicate Order IDs
    .dropDuplicates(["Order_ID"])
)

# Create hash using the CURRENT column names
df_orders_cleaned = df_orders_cleaned.withColumn(
    "Order_Hash",
    sha2(
        concat_ws(
            "|",
            "Order_ID",
            "CustomerID",
            "ProductName",
            "Qty",
            "Order_Date",
            "OrderAmount",
            "Delivery_Status",
            "Payment_Mode",
            "Ship_Address",
            "Email",
            "Promo_Code",
            "Feedback_Score"
        ),
        256
    )
)

display(df_orders_cleaned)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 44, Finished, Available, Finished, False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `CustomerID` cannot be resolved. Did you mean one of the following? [`Order_ID`, `cust_id`, `Email`, `Qty`, `Order_Date`].;
'Project [Order_ID#777, cust_id#778, Product_Name#779, Qty#780, Order_Date#781, Order_Amount$#782, Delivery_Status#783, Payment_Mode#784, Ship_Address#785, Email#786, Promo_Code#787, Feedback_Score#788, trim(upper('CustomerID), None) AS CustomerID#3260]
+- Relation [Order_ID#777,cust_id#778,Product_Name#779,Qty#780,Order_Date#781,Order_Amount$#782,Delivery_Status#783,Payment_Mode#784,Ship_Address#785,Email#786,Promo_Code#787,Feedback_Score#788] parquet


In [43]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

df_orders_cleaned = (
    df_orders

    # Standardize column names
    .withColumnRenamed("cust_id", "CustomerID")
    .withColumnRenamed("Product_Name", "ProductName")
    .withColumnRenamed("Order_Amount$", "OrderAmount")

    # Clean Order Date
    .withColumn(
        "Order_Date",
        coalesce(
            to_date(col("Order_Date"), "dd-MM-yyyy"),
            to_date(col("Order_Date"), "yyyy-MM-dd"),
            to_date(col("Order_Date"), "yyyy/MM/dd"),
            to_date(col("Order_Date"), "dd/MM/yyyy"),
            to_date(col("Order_Date"), "MM/dd/yyyy")
        )
    )

    # Clean Customer ID
    .withColumn(
        "CustomerID",
        trim(upper(col("CustomerID")))
    )

    # Clean Email
    .withColumn(
        "Email",
        lower(trim(col("Email")))
    )

    # Handle empty Payment Mode
    .withColumn(
        "Payment_Mode",
        when(
            length(trim(col("Payment_Mode"))) == 0,
            lit("unknown")
        ).otherwise(col("Payment_Mode"))
    )

    # Replace null Promo Codes
    .withColumn(
        "Promo_Code",
        coalesce(
            col("Promo_Code"),
            lit("NO_PROMO")
        )
    )

    # Clean Order Amount
    .withColumn(
        "OrderAmount",
        abs(
            regexp_extract(
                col("OrderAmount"),
                r"(\d+\.?\d*)",
                1
            ).cast(DoubleType())
        )
    )

    # Add Year and Month
    .withColumn(
        "Order_Year",
        year("Order_Date")
    )
    .withColumn(
        "Order_Month",
        month("Order_Date")
    )

    # Remove rows with missing critical fields
    .dropna(
        subset=[
            "Order_ID",
            "OrderAmount",
            "CustomerID"
        ]
    )

    # Remove duplicate Order IDs
    .dropDuplicates(["Order_ID"])
)

# Create hash AFTER all column renaming
df_orders_cleaned = df_orders_cleaned.withColumn(
    "Order_Hash",
    sha2(
        concat_ws(
            "|",
            "Order_ID",
            "CustomerID",
            "ProductName",
            "Qty",
            "Order_Date",
            "OrderAmount",
            "Delivery_Status",
            "Payment_Mode",
            "Ship_Address",
            "Email",
            "Promo_Code",
            "Feedback_Score"
        ),
        256
    )
)

display(df_orders_cleaned)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 104c12e0-b4b1-4a0e-a265-de4ba827ef61)

In [44]:
df_orders_cleaned.write.mode("overwrite").format("delta").saveAsTable("silver_orders")

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 46, Finished, Available, Finished, False)

In [45]:
from pyspark.sql.functions import *

df_orders = spark.read.table("silver_orders")
df_returns = spark.read.table("silver_returns")
df_inventory = spark.read.table("silver_inventory")

print("Orders:", df_orders.columns)
print("Returns:", df_returns.columns)
print("Inventory:", df_inventory.columns)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 47, Finished, Available, Finished, False)

Orders: ['Order_ID', 'CustomerID', 'ProductName', 'Qty', 'Order_Date', 'OrderAmount', 'Delivery_Status', 'Payment_Mode', 'Ship_Address', 'Email', 'Promo_Code', 'Feedback_Score', 'Order_Year', 'Order_Month', 'Order_Hash']
Returns: ['ReturnID', 'OrderID', 'CustomerID', 'Product', 'ReturnReason', 'ReturnDate', 'RefundStatus', 'PickupAddress', 'ReturnAmount']
Inventory: ['product_id', 'product_name', 'stock', 'warehouse', 'cost_price', 'available', 'last_stocked']


In [46]:
from pyspark.sql.functions import *

# Load Silver tables
df_orders = spark.read.table("silver_orders")
df_returns = spark.read.table("silver_returns")
df_inventory = spark.read.table("silver_inventory")


# =========================================================
# STEP 1: Join Orders with Returns
# =========================================================

df_order_return = (
    df_orders.alias("o")
    .join(
        df_returns.alias("r"),
        col("o.Order_ID") == col("r.OrderID"),
        "left"
    )
)


# =========================================================
# STEP 2: Join Inventory using Product Name
# =========================================================

df_enriched = (
    df_order_return
    .join(
        df_inventory.alias("i"),
        lower(trim(col("o.ProductName"))) ==
        lower(trim(col("i.product_name"))),
        "left"
    )
)


# =========================================================
# STEP 3: Product-level KPIs
# =========================================================

df_kpi = (
    df_enriched
    .groupBy(col("o.ProductName").alias("ProductName"))
    .agg(
        count(col("o.Order_ID")).alias("Total_Orders"),

        countDistinct(
            col("o.CustomerID")
        ).alias("Unique_Customers"),

        count(col("r.ReturnID")).alias("Total_Returns"),

        round(
            (
                count(col("r.ReturnID")) /
                count(col("o.Order_ID"))
            ) * 100,
            2
        ).alias("Return_Rate_Percent"),

        round(
            sum(col("o.OrderAmount")),
            2
        ).alias("Total_Revenue"),

        round(
            avg(col("o.OrderAmount")),
            2
        ).alias("Avg_Order_Value"),

        sum(col("i.stock")).alias("Total_Stock"),

        round(
            avg(col("i.cost_price")),
            2
        ).alias("Avg_Cost"),

        round(
            sum(col("o.OrderAmount")) -
            (
                sum(col("i.stock")) *
                avg(col("i.cost_price"))
            ),
            2
        ).alias("Net_Profit")
    )
)


# =========================================================
# STEP 4: Display Gold KPI table
# =========================================================

display(df_kpi)

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6b3f5ce4-237e-430c-bbc2-266f5053b94c)

In [47]:
df_kpi.write.mode("overwrite").format("delta").saveAsTable("gold_product_kpis")

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 49, Finished, Available, Finished, False)

In [48]:
display(spark.read.table("gold_product_kpis"))

StatementMeta(, ce025987-a1b3-4d4e-a69a-a0c96d528eb1, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4a2113cf-5b5a-4406-bb1a-877a157bf0ad)